# Adjacency Representations



```{contents}
:local:
:depth: 2
```


A graph's abstract idea is vertices plus edges. A program still needs a concrete representation. The two classic choices are an **adjacency matrix** and an **adjacency list**.

The right choice depends on the question you expect to ask. Do you need very fast "is there an edge?" lookups? Do you need to loop over neighbors? Is the graph dense or sparse?


```{index} adjacency matrix
```

## Adjacency Matrices

An adjacency matrix uses a two-dimensional array. Row `i` and column `j` answer whether vertex `i` connects to vertex `j`.


In [ ]:
using System;
using System.Collections.Generic;

string[] vertices = { "A", "B", "C", "D" };
var index = new Dictionary<string, int>();

for (int i = 0; i < vertices.Length; i++)
{
    index[vertices[i]] = i;
}

bool[,] connected = new bool[vertices.Length, vertices.Length];
AddUndirectedEdge("A", "B");
AddUndirectedEdge("A", "C");
AddUndirectedEdge("B", "D");

Console.WriteLine($"A connected to C? {connected[index["A"], index["C"]]}");
Console.WriteLine($"C connected to D? {connected[index["C"], index["D"]]}");

void AddUndirectedEdge(string a, string b)
{
    int row = index[a];
    int col = index[b];
    connected[row, col] = true;
    connected[col, row] = true;
}


The matrix makes edge lookup direct after the vertex names are mapped to indexes. Its space cost is high for sparse graphs because it stores a cell for every possible pair, even when most pairs are not connected.


```{index} weighted adjacency matrix
```

## Weighted Matrices

A weighted matrix can store numbers instead of booleans. A sentinel value can mean "no edge." In production code, a nullable value such as `int?` often communicates that absence more clearly.


In [ ]:
using System;
using System.Collections.Generic;

string[] cities = { "Rolla", "St. Louis", "Chicago" };
var index = new Dictionary<string, int>();

for (int i = 0; i < cities.Length; i++)
{
    index[cities[i]] = i;
}

int?[,] miles = new int?[cities.Length, cities.Length];
AddRoute("Rolla", "St. Louis", 106);
AddRoute("St. Louis", "Chicago", 297);

Console.WriteLine(miles[index["Rolla"], index["Chicago"]] is null
    ? "No direct Rolla-to-Chicago route."
    : "Direct route exists.");

void AddRoute(string from, string to, int distance)
{
    miles[index[from], index[to]] = distance;
    miles[index[to], index[from]] = distance;
}


The nullable matrix keeps edge absence separate from edge weight. That matters because zero may be a meaningful value in some domains.


```{index} adjacency list
```

## Adjacency Lists

An adjacency list stores each vertex with the neighbors it actually has. In C#, a dictionary from vertex names to lists is a natural first representation.


In [ ]:
using System;
using System.Collections.Generic;

var graph = new Dictionary<string, List<string>>
{
    ["A"] = new() { "B", "C" },
    ["B"] = new() { "A", "D" },
    ["C"] = new() { "A" },
    ["D"] = new() { "B" }
};

foreach ((string vertex, List<string> neighbors) in graph)
{
    Console.WriteLine($"{vertex}: {string.Join(", ", neighbors)}");
}


Adjacency lists are compact for sparse graphs because they store actual edges rather than every possible pair. They also make neighbor iteration simple, which is exactly what traversal algorithms need.


```{index} graph representation; sparse graph
```

## Choosing a Representation

A graph is **sparse** when it has relatively few edges compared with the number of possible edges. Most practical relationship graphs are sparse. A graph is **dense** when many pairs of vertices are connected.


In [ ]:
using System;

int vertexCount = 6;
int edgeCount = 5;
int possibleUndirectedEdges = vertexCount * (vertexCount - 1) / 2;
double density = (double)edgeCount / possibleUndirectedEdges;

Console.WriteLine($"Possible undirected edges: {possibleUndirectedEdges}");
Console.WriteLine($"Actual edges: {edgeCount}");
Console.WriteLine($"Density: {density:P1}");

if (density < 0.5)
{
    Console.WriteLine("An adjacency list is usually a natural first choice.");
}
else
{
    Console.WriteLine("An adjacency matrix may be reasonable for frequent edge lookups.");
}


The cutoff in the example is not a universal rule. It is a thinking tool: compare the number of actual edges with the number of possible edges, then match the representation to the operations you need most often.


```{index} graph class
```

## A Small Graph Class

A wrapper class can protect the representation and provide graph operations with clear names.


In [ ]:
using System;
using System.Collections.Generic;

var graph = new SimpleGraph();
graph.AddEdge("A", "B");
graph.AddEdge("A", "C");
graph.AddEdge("B", "D");

foreach (string neighbor in graph.GetNeighbors("A"))
{
    Console.WriteLine($"A is adjacent to {neighbor}");
}

Console.WriteLine($"A connected to D? {graph.HasEdge("A", "D")}");

class SimpleGraph
{
    private readonly Dictionary<string, HashSet<string>> adjacency = new();

    public void AddEdge(string a, string b)
    {
        AddVertex(a);
        AddVertex(b);
        adjacency[a].Add(b);
        adjacency[b].Add(a);
    }

    public bool HasEdge(string a, string b)
    {
        return adjacency.ContainsKey(a) && adjacency[a].Contains(b);
    }

    public IEnumerable<string> GetNeighbors(string vertex)
    {
        return adjacency.TryGetValue(vertex, out HashSet<string>? neighbors)
            ? neighbors
            : Array.Empty<string>();
    }

    private void AddVertex(string vertex)
    {
        adjacency.TryAdd(vertex, new HashSet<string>());
    }
}


The class uses `HashSet<string>` for each neighbor collection, so adding the same edge twice does not duplicate the neighbor. The public methods describe graph behavior while the dictionary remains an implementation detail.


```{rubric} Footnotes
```
[^1]: An undirected adjacency matrix is symmetric: cell `[i, j]` and cell `[j, i]` hold the same connection information.
[^2]: In a directed graph, adjacency lists usually store outgoing neighbors. Some applications also store incoming neighbors for faster reverse lookups.
